In [15]:
import numpy as np
import nnfs
from nnfs.datasets import spiral_data
nnfs.init()

In [16]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons,
                 weight_regularizer_l1=0, weight_regularizer_l2=0,
                 bias_regularizer_l1=0, bias_regularizer_l2=0):
        # Initialize weights and biases
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
        # Set regularization strength
        self.weight_regularizer_l1 = weight_regularizer_l1
        self.weight_regularizer_l2 = weight_regularizer_l2
        self.bias_regularizer_l1 = bias_regularizer_l1
        self.bias_regularizer_l2 = bias_regularizer_l2

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    #backward pass
    def backward(self,dvalues):
        # dvalues - gradient of the loss with respect to the outputs
        # Gradients on parameters
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)

        #gradients on regularization
        # L1
        if self.weight_regularizer_l1 > 0:
            dL1 = np.ones_like(self.weights)
            dL1[self.weights < 0] = -1
            self.dweights = self.dweights + self.weight_regularizer_l1 * dL1

        if self.weight_regularizer_l2 > 0:
            self.dweights = self.dweights + 2 * self.weight_regularizer_l2 * self.weights

        if self.bias_regularizer_l1 > 0:
            dL1 = np.ones_like(self.biases)
            dL1[self.biases < 0] = -1
            self.dbiases = self.dbiases + self.bias_regularizer_l1 * dL1

        if self.bias_regularizer_l2 > 0:
            self.dbiases = self.dbiases + 2 * self.bias_regularizer_l2 * self.biases

        # gradient on values
        # these are derivatives of the loss with respect to inputs
        self.dinputs = np.dot(dvalues, self.weights.T)

In [23]:
class Activation_ReLU:
    def forward(self,inputs):
        self.inputs = inputs
        self.output = np.maximum(0,inputs)

    def backward(self,dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0


In [30]:
class Activation_Softmax:
 # Forward pass
 def forward(self, inputs):
 # Get unnormalized probabilities
  exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
 # Normalize them for each sample
  probabilities = exp_values / np.sum(exp_values, axis=1,keepdims=True)
  self.output = probabilities


In [31]:
# Common loss base class
class Loss:
    # Calculates the data and regularization losses
    # given model output and ground truth values
    def calculate(self, output, y):
        # Calculate sample losses by calling forward implemented in child
        sample_losses = self.forward(output, y)
        # Calculate mean loss
        data_loss = np.mean(sample_losses)
        # Return loss
        return data_loss


In [32]:
class Loss_CategoricalCrossentropy(Loss):
    # Forward pass
    def forward(self, y_pred, y_true):
        # Number of samples in a batch
        samples = len(y_pred)

        # Clip data to prevent division by 0
        # Clip both sides to not drag mean towards any value
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)

        # Probabilities for target values -
        # only if categorical labels
        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[
                range(samples),
                y_true
            ]
        # Mask values - only for one-hot encoded labels
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(
                y_pred_clipped * y_true,
                axis=1
            )

        # Losses
        negative_log_likelihoods = -np.log(correct_confidences)
        return negative_log_likelihoods

    # Backward pass
    def backward(self, dvalues, y_true):
        # Number of samples
        samples = len(dvalues)
        # Number of labels in every sample
        # We'll use the first sample to count them
        labels = len(dvalues[0])

        # If labels are sparse, turn them into one-hot vector
        if len(y_true.shape) == 1:
            y_true = np.eye(labels)[y_true]

        # Calculate gradient
        self.dinputs = -y_true / dvalues
        # Normalize gradient
        self.dinputs = self.dinputs / samples

In [33]:
# Softmax classifier - combined Softmax activation
# and cross-entropy loss for faster backward step
class Activation_Softmax_Loss_CategoricalCrossentropy:
    # Creates activation and loss function objects
    def __init__(self):
        self.activation = Activation_Softmax()
        self.loss = Loss_CategoricalCrossentropy()

    # Forward pass
    def forward(self, inputs, y_true):
        # Output layer's activation function
        self.activation.forward(inputs)
        # Set the output
        self.output = self.activation.output
        # Calculate and return loss value
        return self.loss.calculate(self.output, y_true)

    # Backward pass
    def backward(self, dvalues, y_true):
        # Number of samples
        samples = len(dvalues)
        # If labels are one-hot encoded,
        # turn them into discrete values
        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis=1)
        # Copy so we can safely modify
        self.dinputs = dvalues.copy()
        # Calculate gradient
        self.dinputs[range(samples), y_true] -= 1
        # Normalize gradient
        self.dinputs = self.dinputs / samples

    # def regularization_loss(self,layer):
    #     regularization_loss = 0 

    #     # L1 regularization
    #     if layer.weight_regularizer_l1 > 0:
    #         regularization_loss = regularization_loss + layer.weight_regularizer_l1 * np.sum(np.abs(layer.weights))
    #     if layer.bias_regularizer_l1 > 0:
    #         regularization_loss = regularization_loss + layer.bias_regularizer_l1 * np.sum(np.abs(layer.biases))

    #     # L2 regularization
    #     if layer.weight_regularizer_l2 > 0:
    #         regularization_loss = regularization_loss + layer.weight_regularizer_l2 * np.sum(layer.weights ** 2)
    #     if layer.bias_regularizer_l2 > 0:
    #         regularization_loss = regularization_loss + layer.bias_regularizer_l2 * np.sum(layer.biases ** 2)

    #     return regularization_loss

    # def calculate(self,output, y):
    #     sample_losses = self.forward(output, y)
    #     data_loss = np.mean(sample_losses)
    #     return data_loss

In [34]:
class LossRegularization:
    def regularization_loss(self, layer):
        regularization_loss = 0 

        # L1 regularization
        if layer.weight_regularizer_l1 > 0:
            regularization_loss = regularization_loss + layer.weight_regularizer_l1 * np.sum(np.abs(layer.weights))
        if layer.bias_regularizer_l1 > 0:
            regularization_loss = regularization_loss + layer.bias_regularizer_l1 * np.sum(np.abs(layer.biases))

        # L2 regularization
        if layer.weight_regularizer_l2 > 0:
            regularization_loss = regularization_loss + layer.weight_regularizer_l2 * np.sum(layer.weights ** 2)
        if layer.bias_regularizer_l2 > 0:
            regularization_loss = regularization_loss + layer.bias_regularizer_l2 * np.sum(layer.biases ** 2)

        return regularization_loss

    def calculate(self, output, y):
        sample_losses = self.forward(output, y)
        data_loss = np.mean(sample_losses)
        return data_loss

In [35]:
# Adam optimizer
class Optimizer_Adam:
    # Initialize optimizer - set settings
    def __init__(self, learning_rate=0.001, decay=0., epsilon=1e-7, beta_1=0.9, beta_2=0.999):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.beta_1 = beta_1
        self.beta_2 = beta_2

    # Call once before any parameter updates
    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1. / (1. + self.decay * self.iterations))

    # Update parameters
    def update_params(self, layer):
        # If layer does not contain cache arrays, create them filled with zeros
        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_momentums = np.zeros_like(layer.biases)
            layer.bias_cache = np.zeros_like(layer.biases)

        # Calculate momentum factors (previous weights updates)
        layer.weight_momentums = self.beta_1 * layer.weight_momentums + (1 - self.beta_1) * layer.dweights
        layer.bias_momentums = self.beta_1 * layer.bias_momentums + (1 - self.beta_1) * layer.dbiases

        #calculate numerator
        weight_momentums_corrected = layer.weight_momentums / (1 - self.beta_1 ** (self.iterations + 1))
        bias_momentums_corrected = layer.bias_momentums / (1 - self.beta_1 ** (self.iterations + 1))

        # Calculate weights cache
        layer.weight_cache = self.beta_2 * layer.weight_cache + (1 - self.beta_2) * layer.dweights**2
        layer.bias_cache = self.beta_2 * layer.bias_cache + (1 - self.beta_2) * layer.dbiases**2

        #calculate denominator
        weight_cache_corrected = layer.weight_cache / (1 - self.beta_2 ** (self.iterations + 1))
        bias_cache_corrected = layer.bias_cache / (1 - self.beta_2 ** (self.iterations + 1))

        # Final updation numerator + denominator
        layer.weights = layer.weights - self.current_learning_rate * weight_momentums_corrected / (np.sqrt(weight_cache_corrected) + self.epsilon)
        layer.biases = layer.biases - self.current_learning_rate * bias_momentums_corrected / (np.sqrt(bias_cache_corrected) + self.epsilon)

    # Call once after any parameter updates
    def post_update_params(self):
        self.iterations += 1


In [43]:
X,y = spiral_data(samples=100, classes=3)


# Deeper network: 2 -> 64 -> 64 -> 32 -> 3
dense1 = Layer_Dense(2,64,
                     weight_regularizer_l2=5e-4,
                     bias_regularizer_l2=5e-4)
activation1 = Activation_ReLU()

# Added hidden layer
dense2 = Layer_Dense(64,64,
                     weight_regularizer_l2=5e-4,
                     bias_regularizer_l2=5e-4)
activation2 = Activation_ReLU()

# Added hidden layer
dense3 = Layer_Dense(64,32,
                     weight_regularizer_l2=5e-4,
                     bias_regularizer_l2=5e-4)
activation3 = Activation_ReLU()

# Output layer
dense4 = Layer_Dense(32,3)

# Combined softmax + loss for efficient backward
loss_Activation = Activation_Softmax_Loss_CategoricalCrossentropy()
regularized_loss = LossRegularization()
optimizer = Optimizer_Adam(learning_rate=0.02, decay=5e-5)


for epoch in range(2001):
    # Forward pass through the network
    dense1.forward(X)
    activation1.forward(dense1.output)

    dense2.forward(activation1.output)
    activation2.forward(dense2.output)

    dense3.forward(activation2.output)
    activation3.forward(dense3.output)

    dense4.forward(activation3.output)

    # Use combined forward (activation + loss) to get data loss
    data_loss = loss_Activation.forward(dense4.output, y)

    # Regularization loss across all layers
    regularization_loss = (
        regularized_loss.regularization_loss(dense1) +
        regularized_loss.regularization_loss(dense2) +
        regularized_loss.regularization_loss(dense3) +
        regularized_loss.regularization_loss(dense4)
    )

    loss = data_loss + regularization_loss

    # Predictions & accuracy
    predictions = np.argmax(loss_Activation.output, axis=1)
    if len(y.shape) == 2:
        y_temp = np.argmax(y, axis=1)
    else:
        y_temp = y
    accuracy = np.mean(predictions == y_temp)

    if epoch % 100 == 0:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f} (' +
              f'data_loss: {data_loss:.3f}, ' +
              f'reg_loss: {regularization_loss:.3f}), ' +
              f'lr: {optimizer.current_learning_rate}')

    # Backward pass
    loss_Activation.backward(loss_Activation.output, y)
    dense4.backward(loss_Activation.dinputs)

    activation3.backward(dense4.dinputs)
    dense3.backward(activation3.dinputs)

    activation2.backward(dense3.dinputs)
    dense2.backward(activation2.dinputs)

    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    # Update weights and biases for all layers
    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.update_params(dense3)
    optimizer.update_params(dense4)
    optimizer.post_update_params()


epoch: 0, acc: 0.330, loss: 1.099 (data_loss: 1.099, reg_loss: 0.000), lr: 0.02
epoch: 100, acc: 0.333, loss: 1.099 (data_loss: 1.099, reg_loss: 0.000), lr: 0.019901487636200806
epoch: 100, acc: 0.333, loss: 1.099 (data_loss: 1.099, reg_loss: 0.000), lr: 0.019901487636200806
epoch: 200, acc: 0.333, loss: 1.099 (data_loss: 1.099, reg_loss: 0.000), lr: 0.01980296054260112
epoch: 200, acc: 0.333, loss: 1.099 (data_loss: 1.099, reg_loss: 0.000), lr: 0.01980296054260112
epoch: 300, acc: 0.333, loss: 1.099 (data_loss: 1.099, reg_loss: 0.000), lr: 0.0197054042071038
epoch: 300, acc: 0.333, loss: 1.099 (data_loss: 1.099, reg_loss: 0.000), lr: 0.0197054042071038
epoch: 400, acc: 0.333, loss: 1.099 (data_loss: 1.099, reg_loss: 0.000), lr: 0.01960880435315457
epoch: 400, acc: 0.333, loss: 1.099 (data_loss: 1.099, reg_loss: 0.000), lr: 0.01960880435315457
epoch: 500, acc: 0.333, loss: 1.099 (data_loss: 1.099, reg_loss: 0.000), lr: 0.019513146982779648
epoch: 500, acc: 0.333, loss: 1.099 (data_loss

In [44]:
X_test , y_test = spiral_data(samples=100, classes=3)

dense1.forward(X_test)
activation1.forward(dense1.output)

dense2.forward(activation1.output)
activation2.forward(dense2.output)

dense3.forward(activation2.output)
activation3.forward(dense3.output)

dense4.forward(activation3.output)
loss = loss_Activation.forward(dense4.output, y_test)

predictions = np.argmax(loss_Activation.output, axis=1)
if len(y_test.shape) == 2:
    y_temp = np.argmax(y_test, axis=1)
accuracy = np.mean(predictions == y_temp)
print(f'Validation, acc: {accuracy:.3f}, loss: {loss:.3f}')

Validation, acc: 0.867, loss: 0.614
